# IVIM simulator from vessel network
This notebook is used to simulate the IVIM (Intravoxel Incoherent Motion) signal in vascular networks.

**Main Steps:**
1. **Preprocessing**: Extracting the largest connected component of the network and removing unused parameters to optimize memory usage.
2. **Hydrodynamics**: Calculating flow in individual vessels based on set pressure boundary conditions.
3. **Simulation**: Generating particle trajectories and synthetic IVIM signals.
4. **Tensor Analysis**: Fitting IVIM parameters and then tensors from the resulting signals (isotropic vs. anisotropic diffusion).

**Preprocessing the network**

This part is already done as not preprocessed network would be too large as an attachment

In [ ]:
from scripts.network import VesselNetwork
import os

input_folder = 'networks/'
out_folder = 'cleaned_graphs/'
if not os.path.exists(out_folder):
    os.makedirs(out_folder)

file = 'example_network.json'
print(f"Processing {file}...")
input_path = os.path.join(input_folder, file)
output_path = os.path.join(out_folder, file)

# Load, clean, and save the network
network = VesselNetwork(input_path)
network.clean_network()
print(f"Saving preprocessed network to {output_path}...")
network.save(output_path)

**Compute the flows in each link**

One of four boundary pressure assignment methods has to be chosen

- RadialDistance
- TopologicalDistance
- RadiusBased
- LinearGradient

In [ ]:
from scripts.network import VesselNetwork, RadialDistance, TopologicalDistance, RadiusBased, LinearGradient
import os

input_folder = r'cleaned_graphs/'     
out_folder = r'graphs_velocities/'   
if not os.path.exists(out_folder):
    os.makedirs(out_folder)

file = 'example_network.json'
print(f"Processing {file}...")
input_path = os.path.join(input_folder, file)
output_path = os.path.join(out_folder, file)

#Create VesselNetwork instance
network = VesselNetwork(input_path)

# Choose pressure assigment method
pressure_asigning_strategy = RadialDistance() # RadiusBased()  # TopologicalDistance()# LinearGradient() 

# Calculate flows - choose either target_median of velocity or pressure_difference between nodes with lowest and highest pressure
network.calculate_flows(pressure_asigning_strategy, target_median = 900)

# Perform orientational analysis (saves reference FA and principal direction of vessels in network)
network.orientational_analysis()
# Prints summary of the network
network.print_summary()
# Save netwotk with computed flows and orientations
network.save(output_path)

**Generating and visualization of a single trajectory**

In [ ]:
from scripts.ivim import IVIMSimulator
from scripts.network import VesselNetwork
from scripts.visualize import plot_graph_with_points

# Load network with computed flows and orientations
network_path = r'graphs_velocities/example_network.json'
network = VesselNetwork(network_path)

# Generate and plot trajectory of a single perfusing particle
simulator = IVIMSimulator(network)
perf_trajectory = simulator.generate_n_perfusing_particles(1)
plot_graph_with_points(network.data, perf_trajectory[0])


**Generating and visualization of synthetic IVIM signal in a single gradient direction**

In [ ]:
from scripts.ivim import IVIMSimulator
from scripts.network import VesselNetwork
from scripts.visualize import visualize_result_signal
import numpy as np

out_folder = r'signal_results/'
if not os.path.exists(out_folder):
    os.makedirs(out_folder)

# Load network with computed flows and orientations and create IVIMSimulator instance
network_path = r'graphs_velocities/example_network.json'
network = VesselNetwork(network_path)
simulator = IVIMSimulator(network)

# Note: Generating trajectories may take some time, feel free to lower the number of particles
n_perfusion_particles = 200
n_diffusion_particles = 1000
b_values = np.array([0,20,50,100,200,400,600,800])
ivim_signal = simulator.get_ivim_signal(n_perfusion_particles, n_diffusion_particles, b_values, np.array([1,1,1]), out_file=f'{out_folder}signal.npz')
visualize_result_signal(f'{out_folder}signal.npz')

**Generating IVIM signal in multiple gradient directions, extracting result tensors**

1. Isotropic diffusion:

In [ ]:
from scripts.ivim import IVIMSimulator
from scripts.network import VesselNetwork
import os
import numpy as np

out_folder = r'signal_results/'
if not os.path.exists(out_folder):
    os.makedirs(out_folder)

# Load network with computed flows and orientations and create IVIMSimulator instance
network_path = r'graphs_velocities/example_network.json'
network = VesselNetwork(network_path)
simulator = IVIMSimulator(network)

# Define gradient directions, b-values and number of particles for IVIM signal simulation
gradient_directions = np.array([ [1, 1, 0], [1, -1, 0], [0, 1, 1], [0, 1, -1], [1, 0, 1], [-1, 0, 1] ])
# Note: Generating trajectories may take some time, feel free to lower the number of particles
n_perfusion_particles = 1000
n_diffusion_particles = 5000
b_values = np.array([0,20,50,100,150,200,400,600,800])

# Compute the signals array
ivim_signal = simulator.get_ivim_signal(n_perfusion_particles, n_diffusion_particles, b_values, gradient_directions, out_file=f'{out_folder}signal_array.npz')

# Fit and show result tensors
tensors = simulator.fit_ivim_and_get_tensors(out_path=f'{out_folder}tensors.npz')
simulator.show_tensors()
print("Fitted tensors:")
print("f_tensor:\n", tensors[0])
print("D_star_tensor:\n", tensors[1])
print("D_tensor:\n", tensors[2])
simulator.print_evaluation()

2. Anisotropic diffusion in the direction of the main eigenvector of the orientation tensor

In [ ]:
from scripts.ivim import IVIMSimulator
from scripts.network import VesselNetwork
from scripts.trajectories_utils import get_D_tensor_from_eigvec
import numpy as np

out_folder = r'signal_results/'
if not os.path.exists(out_folder):
    os.makedirs(out_folder)

# Load network with computed flows and orientations
network_path = r'graphs_velocities/example_network.json'
network = VesselNetwork(network_path)

# Create D_tensor from principal direction of vessels in the network and create IVIMSimulator instance - the diffusion will be most prominent in the direction of the vessels
principal_direction = network.data['principal_direction']
D_tensor = get_D_tensor_from_eigvec(principal_direction)

# Create IVIMSimulator instance with custom D_tensor
simulator = IVIMSimulator(network, D_input = D_tensor)

# Define gradient directions, b-values and number of particles for IVIM signal simulation
gradient_directions = np.array([ [1, 1, 0], [1, -1, 0], [0, 1, 1], [0, 1, -1], [1, 0, 1], [-1, 0, 1] ])
# Note: Generating trajectories may take some time, feel free to lower the number of particles
n_perfusion_particles = 1000
n_diffusion_particles = 5000
b_values = np.array([0,20,50,100,150,200,400,600,800])

# Compute the signals array
ivim_signal = simulator.get_ivim_signal(n_perfusion_particles, n_diffusion_particles, b_values, gradient_directions, out_file=f'{out_folder}signal_array.npz')

# Fit and show result tensors
tensors = simulator.fit_ivim_and_get_tensors(out_path=f'{out_folder}tensors.npz')
simulator.show_tensors()
print("Fitted tensors:")
print("f_tensor:\n", tensors[0])
print("D_star_tensor:\n", tensors[1])
print("D_tensor:\n", tensors[2])
simulator.print_evaluation()